In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import math
import random
import numpy as np
import json
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta, MO, TU, WE, TH, FR, SA, SU
import pytz

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# =============================================================================
# ENHANCED DATA GENERATION FOR BUSINESS TASKS
# =============================================================================

# --- Task 1: Invoice Processing (Enhanced) ---
def generate_invoice_data():
    # --- Enhancements ---
    # 1. More varied ID formats.
    # 2. More complex context with dates to enable "OVERDUE" status.
    # 3. More text templates with varied phrasing and structure.
    # 4. Added a "noise" number to challenge the model's amount extraction.

    current_date = datetime(2025, 7, 20)
    paid_ids = {f"INV-{random.randint(2000, 9999)}": current_date - timedelta(days=random.randint(5, 60)) for _ in range(4)}

    context_templates = [
        f"Context: Paid ledger: {list(paid_ids.keys())}",
        f"Reference: Settled invoices are {list(paid_ids.keys())} as of {current_date.strftime('%Y-%m-%d')}.",
    ]
    context = random.choice(context_templates)

    # Decide if the invoice will be PAID, PENDING, or OVERDUE
    rand_choice = random.random()
    if rand_choice < 0.4: # PAID
        invoice_id = random.choice(list(paid_ids.keys()))
        invoice_date = paid_ids[invoice_id]
        status = "PAID"
    elif rand_choice < 0.8: # PENDING
        invoice_id = f"INV-{random.randint(2000, 9999)}"
        while invoice_id in paid_ids: invoice_id = f"INV-{random.randint(2000, 9999)}"
        invoice_date = current_date - timedelta(days=random.randint(1, 29))
        status = "PENDING"
    else: # OVERDUE
        invoice_id = f"INV-{random.randint(2000, 9999)}"
        while invoice_id in paid_ids: invoice_id = f"INV-{random.randint(2000, 9999)}"
        invoice_date = current_date - timedelta(days=random.randint(31, 90))
        status = "OVERDUE"

    amount = round(random.uniform(50.0, 5000.0), 2)
    noise_ref_number = random.randint(10000, 99999)

    templates = [
        f"To Accounts Payable, please process invoice number {invoice_id} dated {invoice_date.strftime('%m-%d-%Y')}. The total due is ${amount:.2f}. Our reference code is {noise_ref_number}.",
        f"Attached is invoice {invoice_id}. The balance is ${amount:.2f}. Please note the date: {invoice_date.strftime('%m/%d/%Y')}.",
        f"RE: Invoice {invoice_id} // Amount: ${amount:.2f}. Dated {invoice_date.strftime('%B %d, %Y')}. This needs processing.",
        f"The amount of ${amount:.2f} for invoice {invoice_id} (dated {invoice_date.strftime('%Y-%m-%d')}) is pending your approval.",
    ]
    text = f"{context}\n---\n{random.choice(templates)}"
    label = {"invoice_id": invoice_id, "total_amount": amount, "status": status}
    return 'invoice', text, json.dumps(label)

# --- Task 2: Support Ticket Analysis (Enhanced) ---
def generate_support_ticket():
    # --- Enhancements ---
    # 1. More email and subject templates.
    # 2. Wider variety of keywords for priority mapping.
    # 3. Order ID can now appear in the subject or body.
    # 4. More nuanced categories.

    emails = ["john.d@email.com", "sara.k@work.net", "test.user@web.org", "help@company.com"]
    categories = ["Billing Inquiry", "Technical Failure", "Product Question", "Contract Renewal"]
    order_id = f"{chr(random.randint(65, 90))}-{random.randint(100,999)}-{chr(random.randint(65, 90))}YZ"

    email = random.choice(emails)
    category = random.choice(categories)

    # Place order_id randomly in subject or body
    subject_text = f"Question about my account"
    body_text = ""
    if random.random() > 0.5:
        subject_text = f"Urgent Issue with order {order_id}"
    else:
        body_text = f"My order number is {order_id}. "

    templates = {
        "Billing Inquiry": (f"I was overcharged on my last invoice. Can we look into it?", "Medium"),
        "Technical Failure": (f"The system is completely down and it's a critical failure!", "High"),
        "Product Question": (f"I have a question about how to use the new feature.", "Low"),
        "Contract Renewal": (f"We need to discuss renewing our enterprise contract for next year.", "Medium"),
    }

    body_addon, priority = templates[category]
    body_text += body_addon

    # Override priority with more specific keywords
    if any(word in body_text.lower() for word in ["critical", "outage", "failure", "cannot work"]):
        priority = "High"
    elif any(word in body_text.lower() for word in ["idea", "feedback", "question"]):
        priority = "Low"

    text = f"From: {email}\nSubject: {subject_text}\n\n{body_text}"
    label = {"category": category, "priority": priority, "customer_email": email, "order_id": order_id}
    return 'ticket', text, json.dumps(label)

# --- Task 3: NL to API Call (Enhanced) ---
def generate_api_call_data():
    # --- Enhancements ---
    # 1. More complex relative time phrases.
    # 2. Expanded system mappings.
    # 3. More command phrasing templates.
    # 4. Introduction of optional boolean parameters in the output JSON.

    now = datetime(2025, 7, 20, 12, 0, 0, tzinfo=pytz.timezone("America/Los_Angeles"))
    actions = ["reboot_server", "run_backup", "deploy_update", "scale_cluster"]
    systems = {"main website": "prod_web_server", "customer db": "customer_db_main", "analytics platform": "analytics_v2", "login service": "auth_service_prod"}

    action = random.choice(actions)
    system_key = random.choice(list(systems.keys()))
    system_val = systems[system_key]

    day_map = {
        "end of day today": now.replace(hour=17, minute=0),
        "tomorrow morning": (now + timedelta(days=1)).replace(hour=9, minute=0),
        "next Friday at noon": now + relativedelta(weekday=FR(1), hour=12, minute=0),
        "in 3 hours": now + timedelta(hours=3)
    }
    time_phrase = random.choice(list(day_map.keys()))
    exec_time = day_map[time_phrase]

    utc_time = exec_time.astimezone(pytz.utc)

    templates = [
        f"I need you to {action.replace('_', ' ')} on the {system_key} {time_phrase}.",
        f"Can you please execute a {action.replace('_', ' ')} for the {system_key}, scheduling it for {time_phrase}?",
        f"Schedule task: {action.replace('_', ' ')}. Target: {system_key}. Time: {time_phrase}.",
    ]
    text = random.choice(templates)

    label = {"action": action, "target_system": system_val, "execute_at_utc": utc_time.strftime("%Y-%m-%dT%H:%M:%SZ")}

    # Add optional parameters
    if action == "reboot_server" and random.random() > 0.5:
        label["force_reboot"] = True
    if action == "deploy_update" and random.random() > 0.5:
        label["notify_channel"] = "#deployments"

    return 'api_call', text, json.dumps(label)


# =============================================================================
# ALL MODEL AND UTILITY CODE (Unchanged)
# The full ZRIA architecture from the previous response goes here.
# =============================================================================
class FractalAttentionalResonance(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.bias_generator = nn.Sequential(nn.Linear(dim, dim // 2), nn.ReLU(), nn.Linear(dim // 2, self.num_heads * self.head_dim))
    def forward(self, x, mask=None):
        B, T, D = x.shape
        global_context = x.mean(dim=1)
        dynamic_bias = self.bias_generator(global_context).view(B, self.num_heads, self.head_dim)
        Q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.head_dim)
        bias = dynamic_bias.unsqueeze(-1)
        fractal_resonance = torch.matmul(Q, bias)
        scores = scores + fractal_resonance
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        context = torch.matmul(F.softmax(scores, dim=-1), V)
        context = context.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(context)

class FAR_TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.self_attn = FractalAttentionalResonance(d_model, nhead)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, src, src_mask=None):
        src2 = self.self_attn(src, mask=src_mask)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(F.gelu(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

class CustomTransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([FAR_TransformerEncoderLayer(d_model, nhead, dim_feedforward) for _ in range(num_layers)])
    def forward(self, src, mask=None):
        for layer in self.layers:
            src = layer(src, src_mask=mask)
        return src

class FractalEmbeddingLayer(nn.Module):
    def __init__(self, dim, num_fractals=4):
        super().__init__()
        self.num_fractals = num_fractals
        self.dims = nn.Parameter(torch.rand(num_fractals) * 2 + 1)
        self.weight_generator = nn.Sequential(nn.Linear(dim, dim // 2), nn.ReLU(), nn.Linear(dim // 2, num_fractals))
        self.fractal_functions = [lambda x: torch.sin(x*2*math.pi), lambda x: x-torch.floor(x), lambda x: 4*x*(1-x), lambda x: torch.sigmoid(5*(x-0.5))]
    def forward(self, x):
        x_safe = torch.sigmoid(x)
        p_weights = F.softmax(self.weight_generator(x.mean(dim=1)), dim=-1).unsqueeze(1).unsqueeze(-1)
        fractal_outputs = []
        for i in range(self.num_fractals):
            output = self.fractal_functions[i](torch.pow(x_safe, 1.0 / self.dims[i]))
            fractal_outputs.append(output.unsqueeze(-2))
        weighted_fractals = p_weights * torch.cat(fractal_outputs, dim=-2)
        return x + torch.sum(weighted_fractals, dim=-2)

class DynamicGating(nn.Module):
    def __init__(self, dim, num_experts=4):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(dim, num_experts), nn.Softmax(dim=-1))
        self.experts = nn.ModuleList([nn.Linear(dim, dim) for _ in range(num_experts)])
    def forward(self, x):
        gates = self.gate(x).unsqueeze(-2)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=-1)
        return torch.sum(expert_outputs * gates, dim=-1)

class ContinuousResonanceField(nn.Module):
    def __init__(self, dim, nhead=4):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.GELU(), nn.LayerNorm(dim))
        self.attn = nn.MultiheadAttention(dim, nhead, batch_first=True)
    def forward(self, query, context):
        field_context = self.mlp(context)
        attended, _ = self.attn(query, field_context, field_context)
        return attended

class QuantumInspiredFusion(nn.Module):
    def __init__(self, dim, num_qubits=4):
        super().__init__()
        self.num_qubits = num_qubits
        self.prep_k = nn.Linear(dim, num_qubits * 2)
        self.prep_e = nn.Linear(dim, num_qubits * 2)
        self.gate = nn.Parameter(torch.randn(num_qubits, num_qubits))
        self.measure = nn.Linear(num_qubits * 2, dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, know, exec):
        B, T, _ = know.shape
        k_state = self.prep_k(know).view(B, T, self.num_qubits, 2)
        e_state = self.prep_e(exec).view(B, T, self.num_qubits, 2)
        k_real, k_imag = k_state[..., 0], k_state[..., 1]
        e_real, e_imag = e_state[..., 0], e_state[..., 1]
        ent_real = torch.matmul(k_real * e_real - k_imag * e_imag, self.gate)
        ent_imag = torch.matmul(k_real * e_imag + k_imag * e_real, self.gate)
        state = torch.stack([ent_real, ent_imag], dim=-1).reshape(B, T, -1)
        return self.norm(self.measure(state))

class HierarchicalMemorySystem(nn.Module):
    def __init__(self, dim, scales=[32, 16, 8], nhead=4):
        super().__init__()
        self.banks = nn.ParameterList([nn.Parameter(torch.randn(1, s, dim)) for s in scales])
        self.projs = nn.ModuleList([nn.Linear(dim, dim) for _ in range(len(scales) * 3)])
        self.gates = nn.ModuleList([nn.Linear(dim * 2, dim) for _ in scales])
        self.scale_attn = nn.MultiheadAttention(dim, nhead, batch_first=True)
        self.gru = nn.GRU(dim, dim, batch_first=True)
    def forward(self, x):
        B, T, D = x.shape
        scale_outputs = []
        for i, memory in enumerate(self.banks):
            mem = memory.expand(B, -1, -1)
            q, k, v = self.projs[i*3](x), self.projs[i*3+1](mem), self.projs[i*3+2](mem)
            attended_mem = F.softmax(torch.bmm(q, k.transpose(1, 2)) / math.sqrt(D), dim=-1) @ v
            gate = torch.sigmoid(self.gates[i](torch.cat([x, attended_mem], dim=-1)))
            scale_outputs.append(x * (1 - gate) + attended_mem * gate)
        fused, _ = self.scale_attn(x, torch.cat(scale_outputs, dim=1), torch.cat(scale_outputs, dim=1))
        consolidated, _ = self.gru(fused)
        return consolidated

class NeuralODE(nn.Module):
    def __init__(self, dim, steps=4):
        super().__init__()
        self.steps = steps
        self.net = nn.Sequential(nn.Linear(dim, dim*2), nn.Tanh(), nn.Linear(dim*2, dim))
    def forward(self, x):
        state = x
        dt = 1.0 / self.steps
        for _ in range(self.steps):
            state = state + dt * self.net(state)
        return state

class ZRIA_DecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.self_attn = FractalAttentionalResonance(d_model, nhead)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        tgt2 = self.self_attn(tgt, mask=tgt_mask)
        tgt = self.norm1(tgt + self.dropout1(tgt2))
        tgt2, _ = self.cross_attn(tgt, memory, memory, key_padding_mask=memory_mask)
        tgt = self.norm2(tgt + self.dropout2(tgt2))
        tgt2 = self.linear2(self.dropout(F.gelu(self.linear1(tgt))))
        tgt = self.norm3(tgt + self.dropout3(tgt2))
        return tgt

class ZRIA_Decoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([ZRIA_DecoderLayer(d_model, nhead, dim_feedforward) for _ in range(num_layers)])
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        output = tgt
        for layer in self.layers:
            output = layer(output, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)
        return output

class ZRIA_Agent(nn.Module):
    def __init__(self, vocab_size, dim, max_seq_len, num_tasks, nhead=4, num_layers=2):
        super().__init__()
        self.dim = dim
        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.positional_embedding = nn.Parameter(torch.randn(1, max_seq_len, dim))
        self.task_embedding = nn.Embedding(num_tasks, dim)
        self.fractal_embedding = FractalEmbeddingLayer(dim)
        self.knowledge_encoder = CustomTransformerEncoder(dim, nhead, dim*2, num_layers)
        self.execution_encoder = CustomTransformerEncoder(dim, nhead, dim*2, num_layers)
        self.know_to_exec_cross_attn = nn.MultiheadAttention(dim, nhead, batch_first=True)
        self.exec_to_know_cross_attn = nn.MultiheadAttention(dim, nhead, batch_first=True)
        self.cross_norm_k = nn.LayerNorm(dim)
        self.cross_norm_e = nn.LayerNorm(dim)
        self.dynamic_gating = DynamicGating(dim)
        self.resonance_field = ContinuousResonanceField(dim)
        self.quantum_fusion = QuantumInspiredFusion(dim)
        self.hierarchical_memory = HierarchicalMemorySystem(dim)
        self.neural_ode = NeuralODE(dim)
        self.fusion_norm = nn.LayerNorm(dim)
        self.decoder = ZRIA_Decoder(dim, nhead, dim*2, num_layers)
        self.output_head = nn.Linear(dim, vocab_size)

    def encode(self, src_ids, task_idx, src_mask):
        task_emb = self.task_embedding(task_idx).unsqueeze(1)
        src_emb = self.token_embedding(src_ids) + self.positional_embedding[:, :src_ids.shape[1], :] + task_emb
        pfaf_x = self.fractal_embedding(src_emb)
        know_path = self.knowledge_encoder(pfaf_x, mask=src_mask)
        exec_path = self.execution_encoder(pfaf_x, mask=src_mask)
        exec_from_know, _ = self.know_to_exec_cross_attn(query=exec_path, key=know_path, value=know_path)
        exec_path = self.cross_norm_e(exec_path + exec_from_know)
        know_from_exec, _ = self.exec_to_know_cross_attn(query=know_path, key=exec_path, value=exec_path)
        know_path = self.cross_norm_k(know_path + know_from_exec)
        base = know_path + exec_path
        fused = self.dynamic_gating(base) + base
        fused = self.resonance_field(fused, base) + fused
        fused = self.quantum_fusion(fused, base) + fused
        fused = self.hierarchical_memory(fused) + fused
        memory = self.neural_ode(fused) + fused
        return self.fusion_norm(memory)

    def decode(self, tgt_ids, memory, tgt_mask, memory_mask=None):
        tgt_emb = self.token_embedding(tgt_ids) + self.positional_embedding[:, :tgt_ids.shape[1], :]
        decoded_out = self.decoder(tgt_emb, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)
        return self.output_head(decoded_out)

    def forward(self, src_ids, task_idx, tgt_ids):
        src_mask = (src_ids != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt_ids != 0).unsqueeze(1).unsqueeze(2)
        causal_mask = torch.tril(torch.ones(tgt_ids.shape[1], tgt_ids.shape[1], device=src_ids.device)).bool()
        tgt_mask = tgt_mask & causal_mask
        memory = self.encode(src_ids, task_idx, src_mask)
        logits = self.decode(tgt_ids, memory, tgt_mask)
        return logits

    def generate(self, src_ids, task_idx, tokenizer, max_len=128):
        self.eval()
        device = src_ids.device
        with torch.no_grad():
            src_mask = (src_ids != 0).unsqueeze(1).unsqueeze(2)
            memory = self.encode(src_ids, task_idx, src_mask)
            output_tokens = torch.tensor([tokenizer.sos_idx], device=device).unsqueeze(0)
            for _ in range(max_len):
                tgt_mask = torch.tril(torch.ones(output_tokens.shape[1], output_tokens.shape[1], device=device)).bool()
                logits = self.decode(output_tokens, memory, tgt_mask.unsqueeze(0).unsqueeze(0))
                next_token = logits[:, -1, :].argmax(dim=-1)
                if next_token.item() == tokenizer.eos_idx: break
                output_tokens = torch.cat([output_tokens, next_token.unsqueeze(0)], dim=1)
        return tokenizer.decode(output_tokens[0].cpu().numpy())

class AgenticTokenizer:
    def __init__(self, corpus):
        special_tokens = ['<PAD>', '<UNK>', '<SOS>', '<EOS>']
        all_chars = sorted(list(set("".join(corpus) + "{}[]:,\"")))
        self.char_to_idx = {t: i for i, t in enumerate(special_tokens)}
        for char in all_chars:
            if char not in self.char_to_idx: self.char_to_idx[char] = len(self.char_to_idx)
        self.idx_to_char = {i: c for c, i in self.char_to_idx.items()}
        self.vocab_size = len(self.char_to_idx)
        self.pad_idx, self.sos_idx, self.eos_idx = 0, 2, 3
    def encode(self, text, is_target=False):
        tokens = [self.char_to_idx.get(c, 1) for c in text]
        return [self.sos_idx] + tokens + [self.eos_idx] if is_target else tokens
    def decode(self, tokens):
        return "".join([self.idx_to_char.get(t, '') for t in tokens]).replace('<SOS>', '').replace('<EOS>', '')

class AgenticDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data, self.tokenizer, self.max_len = data, tokenizer, max_len
        self.task_map = {'invoice': 0, 'ticket': 1, 'api_call': 2}
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        task_type, text, label_json = self.data[idx]
        src_tokens = self.tokenizer.encode(text)
        tgt_tokens = self.tokenizer.encode(label_json, is_target=True)
        src_padded = src_tokens + [self.tokenizer.pad_idx] * (self.max_len - len(src_tokens))
        tgt_padded = tgt_tokens + [self.tokenizer.pad_idx] * (self.max_len - len(tgt_tokens))
        return {'src_ids': torch.tensor(src_padded[:self.max_len]), 'tgt_ids': torch.tensor(tgt_padded[:self.max_len]), 'task_idx': torch.tensor(self.task_map[task_type]), 'label_json': label_json}

def evaluate(model, dataloader, tokenizer, device):
    model.eval()
    total_em, total_f1, count = 0, 0, 0
    with torch.no_grad():
        for batch in dataloader:
            src_ids, task_idx, label_json_list = batch['src_ids'].to(device), batch['task_idx'].to(device), batch['label_json']
            for i in range(src_ids.shape[0]):
                generated_text = model.generate(src_ids[i].unsqueeze(0), task_idx[i].unsqueeze(0), tokenizer)
                try:
                    pred_json = json.loads(generated_text)
                    true_json = json.loads(label_json_list[i])
                    if pred_json == true_json: total_em += 1
                    pred_items, true_items = set(pred_json.items()), set(true_json.items())
                    tp = len(pred_items.intersection(true_items))
                    precision = tp / len(pred_items) if len(pred_items) > 0 else 0
                    recall = tp / len(true_items) if len(true_items) > 0 else 0
                    total_f1 += 2*(precision*recall)/(precision+recall) if (precision+recall)>0 else 0
                except (json.JSONDecodeError, TypeError):
                    total_f1 += 0
                count += 1
    return total_em / count, total_f1 / count

# =============================================================================
# MAIN EXECUTION BLOCK (WITH MORE DATA)
# =============================================================================
if __name__ == '__main__':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # Increase dataset size from 600 to 3000
    print("Generating a larger and more varied dataset (3000 samples)...")
    dataset_raw = [f() for f in [generate_invoice_data, generate_support_ticket, generate_api_call_data] for _ in range(1000)]
    random.shuffle(dataset_raw)

    # Using a standard 80/20 split is not feasible here due to data generation, so we'll do 2500/500
    train_data, val_data = dataset_raw[:2500], dataset_raw[2500:]
    print(f"Dataset created. Training samples: {len(train_data)}, Validation samples: {len(val_data)}")

    corpus = [t for _, t, _ in dataset_raw] + [l for _, _, l in dataset_raw]
    tokenizer = AgenticTokenizer(corpus)
    train_dataset = AgenticDataset(train_data, tokenizer)
    val_dataset = AgenticDataset(val_data, tokenizer)
    train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=8)

    model = ZRIA_Agent(vocab_size=tokenizer.vocab_size, dim=128, max_seq_len=256, num_tasks=3).to(device)
    print(f"Model: Full ZRIA Agent (Encoder-Decoder)")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    optimizer = optim.AdamW(model.parameters(), lr=5e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)
    epochs = 25

    print("\n=== Starting Full ZRIA Agent Training with Enhanced Data ===")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_dataloader:
            src_ids, tgt_ids, task_idx = batch['src_ids'].to(device), batch['tgt_ids'].to(device), batch['task_idx'].to(device)
            optimizer.zero_grad()
            logits = model(src_ids, task_idx, tgt_ids[:, :-1])
            loss = criterion(logits.view(-1, logits.size(-1)), tgt_ids[:, 1:].reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_dataloader)
        em_score, f1_score = evaluate(model, val_dataloader, tokenizer, device)
        print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f} | Val EM: {em_score:.3f} | Val F1: {f1_score:.3f}")

    print("\n=== Inference Demo on Final Model ===")
    for i in range(3):
        sample = val_dataset[i]
        text = tokenizer.decode(sample['src_ids'].numpy())
        generated_json_str = model.generate(sample['src_ids'].unsqueeze(0).to(device), sample['task_idx'].unsqueeze(0).to(device), tokenizer)
        print(f"\n--- TASK: {list(val_dataset.task_map.keys())[sample['task_idx']]} ---\nINPUT:\n{text.strip()}\n\nEXPECTED:\n{sample['label_json']}\n\nMODEL GENERATED:\n{generated_json_str}\n" + "-"*20)

Using device: cuda
Generating a larger and more varied dataset (3000 samples)...
Dataset created. Training samples: 2500, Validation samples: 500
Model: Full ZRIA Agent (Encoder-Decoder)
Trainable parameters: 1,861,954

=== Starting Full ZRIA Agent Training with Enhanced Data ===
Epoch 1/25 | Avg Loss: 0.9526 | Val EM: 0.000 | Val F1: 0.160
Epoch 2/25 | Avg Loss: 0.1893 | Val EM: 0.070 | Val F1: 0.432
Epoch 3/25 | Avg Loss: 0.1487 | Val EM: 0.102 | Val F1: 0.526
Epoch 4/25 | Avg Loss: 0.1384 | Val EM: 0.156 | Val F1: 0.524
Epoch 5/25 | Avg Loss: 0.1346 | Val EM: 0.182 | Val F1: 0.553
Epoch 6/25 | Avg Loss: 0.1319 | Val EM: 0.240 | Val F1: 0.589
Epoch 7/25 | Avg Loss: 0.1233 | Val EM: 0.210 | Val F1: 0.513
Epoch 8/25 | Avg Loss: 0.1162 | Val EM: 0.222 | Val F1: 0.558
Epoch 9/25 | Avg Loss: 0.1136 | Val EM: 0.186 | Val F1: 0.530
Epoch 10/25 | Avg Loss: 0.1037 | Val EM: 0.214 | Val F1: 0.508
Epoch 11/25 | Avg Loss: 0.0969 | Val EM: 0.236 | Val F1: 0.543
Epoch 12/25 | Avg Loss: 0.0884 | Va